# 🧠 Physics-Informed Neural Networks (PINNs)
## Tutorial 0 — Background, Intuition & Setup

---

> **This is your pre-flight checklist.**  
> No heavy math. No training loops. Just the mental model and the environment you need before Tutorial 1.

---

## 1 — What is a PINN, in one paragraph?

A **Physics-Informed Neural Network** is a neural network trained to satisfy a **partial differential equation (PDE)** rather than a labelled dataset.

Instead of minimising `prediction vs. ground truth`, it minimises:

$$
\mathcal{L} = \underbrace{\|\text{PDE residual}\|^2}_{\text{physics}} + \underbrace{\|\text{BC error}\|^2}_{\text{boundary conditions}}
$$

The network learns a function $u(x)$ — or $u(x,y,t)$ — that simultaneously:
- satisfies the governing equation everywhere in the domain
- matches the boundary and initial conditions

**No mesh. No labelled simulation data. Just the equation.**

---

## 2 — Classical Solver vs. PINN: Side-by-Side

| | Classical CFD / FEA | PINN |
|---|---|---|
| **Domain discretisation** | Mesh (cells/nodes) | Collocation points (scatter) |
| **Derivatives** | Finite differences / FVM | Automatic differentiation (exact) |
| **Training data needed** | No | No (only PDE + BCs) |
| **Handles irregular geometry** | Hard (meshing cost) | Easy (just sample points) |
| **Scales to high Re / turbulence** | Yes (established) | Active research area |
| **Inverse problems** | Hard | Native (add parameters to network) |
| **Speed at inference** | Re-solve for every config | Forward pass only (milliseconds) |
| **Accuracy** | Very high (mature) | Problem-dependent, improving |

> **Bottom line**: PINNs are not replacing OpenFOAM tomorrow. They are a powerful tool for inverse problems, parameter estimation, surrogate modelling, and problems where data is sparse.

---

## 3 — The Key Idea: Automatic Differentiation

The reason PINNs work is **autograd** — PyTorch can compute **exact** derivatives of the network output with respect to its inputs.

```
Classical FD:    du/dx ≈ [u(x+h) - u(x)] / h      ← approximate, h must be small
Autograd:        du/dx = exact chain-rule derivative ← exact, no grid needed
```

This means if the network produces $\hat{u}(x)$, we can compute:
- $\frac{d\hat{u}}{dx}$ — first derivative
- $\frac{d^2\hat{u}}{dx^2}$ — second derivative
- $\frac{\partial \hat{u}}{\partial t}$, $\nabla^2 \hat{u}$, etc.

All of these can be plugged directly into the PDE to form the **residual**.

Here is the minimal autograd demo — the most important 10 lines in this series:

In [ ]:
import torch

# A single input point x = 0.5, we want to differentiate w.r.t. it
x = torch.tensor([0.5], requires_grad=True)

# Define a simple function: u = sin(pi * x)
u = torch.sin(torch.pi * x)

# First derivative: du/dx via autograd
du_dx = torch.autograd.grad(u, x, create_graph=True)[0]

# Second derivative: d²u/dx²
d2u_dx2 = torch.autograd.grad(du_dx, x)[0]

# Analytical values for comparison
import math
print(f'x               = {x.item():.4f}')
print(f'u(x)            = {u.item():.6f}    (exact: {math.sin(math.pi * 0.5):.6f})')
print(f'du/dx (autograd)= {du_dx.item():.6f}    (exact: {math.pi * math.cos(math.pi * 0.5):.6f})')
print(f'd²u/dx²(autograd)= {d2u_dx2.item():.6f}   (exact: {-(math.pi**2) * math.sin(math.pi * 0.5):.6f})')

> **What you just saw**: PyTorch computed exact second derivatives with 2 lines. No finite difference stencil. This is the engine of every PINN.

---

## 4 — The PINN Recipe (applies to every tutorial in this series)

```
Step 1 — Write down the PDE
         e.g.  -u'' = f(x),   u(0) = 0,   u(1) = 0

Step 2 — Sample collocation points
         Interior points: enforce the PDE
         Boundary points: enforce BCs / ICs

Step 3 — Build a neural network  u_pred = NN(x; θ)
         Input:  spatial / temporal coordinates
         Output: solution field u

Step 4 — Define the loss
         L_PDE = mean( r(x)² )          r = PDE residual
         L_BC  = mean( (u_pred - u_bc)² )
         L     = L_PDE + λ * L_BC

Step 5 — Minimize L w.r.t. θ  (Adam or L-BFGS)

Step 6 — Evaluate and validate
```

That's it. The same six steps carry you from 1D ODE (Tutorial 1) all the way to 2D Navier-Stokes (Tutorial 4).

---

## 5 — Why `tanh` and Not `ReLU`?

To compute $d^2u/dx^2$ from the network, the activation function must be **twice differentiable**.

| Activation | Differentiable | 2nd derivative | Use in PINN? |
|---|---|---|---|
| `tanh` | ✅ everywhere | ✅ smooth | ✅ default choice |
| `sin` (SIREN) | ✅ everywhere | ✅ smooth | ✅ great for oscillatory PDEs |
| `ReLU` | ✅ (except x=0) | ❌ zero almost everywhere | ❌ avoid |
| `Sigmoid` | ✅ everywhere | ✅ smooth | ⚠️ vanishing gradients |
| `GELU` | ✅ everywhere | ✅ smooth | ✅ works, less common |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_np = np.linspace(-3, 3, 400)

def tanh_fn(x):  return np.tanh(x)
def relu_fn(x):  return np.maximum(0, x)
def tanh_d2(x):  return -2 * np.tanh(x) * (1 - np.tanh(x)**2)  # d²/dx² tanh
def relu_d2(x):  return np.zeros_like(x)                          # d²/dx² ReLU = 0

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(x_np, tanh_fn(x_np), 'b-', lw=2, label='tanh')
axes[0].plot(x_np, relu_fn(x_np), 'r--', lw=2, label='ReLU')
axes[0].set_title('Activation Functions'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_np, tanh_d2(x_np), 'b-', lw=2, label="tanh'' (non-zero)")
axes[1].plot(x_np, relu_d2(x_np), 'r--', lw=2, label="ReLU'' = 0 everywhere")
axes[1].set_title('Second Derivatives — Why ReLU Breaks PINNs')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6 — What Are Collocation Points?

Collocation points are just **locations in the domain where we check the physics**. They are not a mesh — they have no connectivity, no neighbour relationships.

```
FVM mesh:          ┌──┬──┬──┬──┬──┐   cells connected, fluxes between them
                   └──┴──┴──┴──┴──┘

PINN collocation:  ·  ·  ·  ·  ·  ·   independent points, just (x, y) coordinates
```

You can sample them:
- **Uniformly** — `torch.linspace` for 1D, `torch.meshgrid` for 2D
- **Randomly (Latin Hypercube)** — better coverage in high dimensions
- **Adaptively** — more points where the residual is large (advanced)

In [ ]:
# Three sampling strategies side by side — visual demo for the video

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
N = 100

# 1. Uniform grid
x1 = np.linspace(0, 1, int(np.sqrt(N)))
xg, yg = np.meshgrid(x1, x1)
axes[0].scatter(xg, yg, s=12, c='steelblue', alpha=0.8)
axes[0].set_title(f'Uniform Grid\n({int(np.sqrt(N))}×{int(np.sqrt(N))} = {int(np.sqrt(N))**2} pts)')
axes[0].set_xlim(0,1); axes[0].set_ylim(0,1); axes[0].set_aspect('equal')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# 2. Random uniform
np.random.seed(42)
xr = np.random.rand(N); yr = np.random.rand(N)
axes[1].scatter(xr, yr, s=12, c='tomato', alpha=0.8)
axes[1].set_title(f'Random Uniform\n({N} pts)')
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1); axes[1].set_aspect('equal')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

# 3. Latin Hypercube (manual implementation)
def latin_hypercube(n, d=2):
    pts = np.zeros((n, d))
    for i in range(d):
        pts[:, i] = (np.random.permutation(n) + np.random.rand(n)) / n
    return pts

lhs = latin_hypercube(N)
axes[2].scatter(lhs[:,0], lhs[:,1], s=12, c='seagreen', alpha=0.8)
axes[2].set_title(f'Latin Hypercube\n({N} pts, better coverage)')
axes[2].set_xlim(0,1); axes[2].set_ylim(0,1); axes[2].set_aspect('equal')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')

plt.suptitle('Collocation Point Sampling Strategies', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 7 — Environment Setup

Everything in this series runs with **4 packages only**.

```bash
pip install torch numpy matplotlib scipy
```

Or with conda:
```bash
conda create -n pinns python=3.10
conda activate pinns
conda install pytorch numpy matplotlib scipy -c pytorch
```

Let's verify everything is in order:

In [ ]:
import torch
import numpy as np
import matplotlib
import scipy

print('Package versions')
print('-' * 30)
print(f'PyTorch   : {torch.__version__}')
print(f'NumPy     : {np.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'SciPy     : {scipy.__version__}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice    : {device}')
if device == 'cuda':
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
else:
    print('GPU       : not available — CPU is fine for Tutorial 1 & 2')

print('\n✅  All good. You are ready for Tutorial 1.')

---

## 8 — Tutorial Series Roadmap

| Tutorial | Problem | Key concept introduced |
|---|---|---|
| **0 (this)** | Setup & background | Autograd, collocation, recipe |
| **1** | 1D Poisson: $-u'' = f$ | PINN from scratch, BC loss, loss plots |
| **2** | 1D Burgers (steady, nonlinear) | Nonlinear residuals, L-BFGS, adaptive sampling |
| **3** | 2D Poisson | `meshgrid` collocation, 2D error maps |
| **4** | 2D Navier-Stokes (lid-driven cavity) | Vector-valued output, pressure, incompressibility |

---

## 9 — Common Failure Modes (and how to fix them)

Know these before Tutorial 1 so you recognise them when they happen:

| Symptom | Likely cause | Fix |
|---|---|---|
| Loss doesn't decrease | LR too high or too low | Try `1e-3` first, then `1e-4` |
| BCs not satisfied | `λ_BC` too small | Increase to 10–100 |
| Good loss, wrong solution | Network found a trivial solution (e.g. u=0) | Check `λ_BC`, check BCs are correctly coded |
| NaN loss | Exploding gradients | Reduce LR, add gradient clipping |
| High error near boundaries | Collocation points not covering boundary | Add explicit BC points |
| Slow convergence | Adam stuck in flat region | Switch to L-BFGS after Adam (Tutorial 2) |

---

## 10 — Key Vocabulary Cheat Sheet

```
PDE               Partial Differential Equation — the governing physics
BVP               Boundary Value Problem — PDE + boundary conditions
BC                Boundary Condition — what u must equal at the edges
IC                Initial Condition — what u must equal at t=0 (time-dependent)
Residual          How much the network currently violates the PDE
Collocation pts   Points where we evaluate and penalise the residual
Autograd          Automatic differentiation — exact derivatives via chain rule
L_PDE             Loss term from PDE residual
L_BC              Loss term from boundary condition mismatch
λ (lambda)        Weight balancing L_PDE vs L_BC
Adam              Adaptive moment estimation — default optimizer for PINNs
L-BFGS            Second-order optimizer — used for fine-tuning after Adam
L2 error          Root mean squared error vs exact solution — accuracy metric
tanh              Hyperbolic tangent — preferred activation for PINNs
```

---

**You are ready. Open Tutorial 1. ➡️**